## Quickstart: Role-Playing Dimensions in Databricks

This notebook demonstrates how to use the `f_create_roleplaying_global_temp_view` utility to create role-playing dimensions from a sample CSV file (`dim_date_sample.csv`).

**Prerequisites:**
1. This project should be cloned into Databricks Repos.
2. This notebook should be located in the `examples` folder.
3. A cluster must be attached to the notebook.

### 1. Import Utility Function

To make our project's source code importable within the notebook, we add its directory to the system path.

In [0]:
import sys
import os
# This allows the notebook to import from the 'src' folder as if it were an installed package.
sys.path.append(os.path.abspath('../src'))

from roleplay import f_create_roleplaying_global_temp_view

print("Successfully imported 'f_create_roleplaying_global_temp_view' from src/roleplay.py")

Successfully imported 'f_create_roleplaying_global_temp_view' from src/roleplay.py


### 2. Load Sample Data

Next, we load the sample data from `dim_date_sample.csv` into a Spark DataFrame. The `spark` session is automatically available as a global variable in a Databricks notebook.

In [0]:
# Load data using the provided path.
df_dim_date = spark.read.option("header", "true").option("inferSchema", "true").csv("/mnt/manualupload/temp/dim_date_sample.csv")

print("Sample dimension data loaded:")
df_dim_date.show()
df_dim_date.printSchema()

Sample dimension data loaded:
+--------+----------+----------------+----------+----------+
|date_key| full_date|day_of_week_name|is_weekend|is_holiday|
+--------+----------+----------------+----------+----------+
|20250101|2025-01-01|       Wednesday|     false|      true|
|20250102|2025-01-02|        Thursday|     false|     false|
|20250103|2025-01-03|          Friday|     false|     false|
|20250104|2025-01-04|        Saturday|      true|     false|
|20250105|2025-01-05|          Sunday|      true|     false|
|20250106|2025-01-06|          Monday|     false|     false|
|20250107|2025-01-07|         Tuesday|     false|     false|
+--------+----------+----------------+----------+----------+

root
 |-- date_key: integer (nullable = true)
 |-- full_date: date (nullable = true)
 |-- day_of_week_name: string (nullable = true)
 |-- is_weekend: boolean (nullable = true)
 |-- is_holiday: boolean (nullable = true)



### 3. Define Role Map and Create Views

Here, we define the different roles for our date dimension. The utility automatically prefixes all columns with the role's name. We can provide explicit mappings in the `columns` dictionary to override this for specific columns.

- **order_date**: `date_key` and `full_date` are explicitly mapped. Other columns (`day_of_week_name`, `is_weekend`, etc.) will be automatically prefixed.
- **ship_date**: Same logic as `order_date`, but with an added filter to exclude weekends.

In [0]:
role_map = {
    "order_date": {
        "columns": {
            "date_key": "order_date_key",      # Override automatic prefixing for the key
            "full_date": "order_date"           # Override for a cleaner main date column
        }
    },
    "ship_date": {
        "columns": {
            "date_key": "ship_date_key",
            "full_date": "ship_date"
        },
        "filter": "is_weekend = false"
    }
}

# The main function call. We set `p_use_global_views=False` to create session-level
# temp views.
f_create_roleplaying_global_temp_view(
    p_spark=spark,
    p_base_df=df_dim_date,
    p_role_map=role_map,
    p_view_prefix="rp_",
    p_use_global_views=False
)

2025-11-07 07:20:47,699 INFO roleplay: Created session temp view: rp_order_date (cols: order_date_key, order_date, order_date_day_of_week_name, order_date_is_weekend, order_date_is_holiday)
2025-11-07 07:20:47,804 INFO roleplay: Created session temp view: rp_ship_date (cols: ship_date_key, ship_date, ship_date_day_of_week_name, ship_date_is_weekend, ship_date_is_holiday)


### 4. Verify View Creation

Let's query the newly created temporary views. Notice how `rp_ship_date` correctly excludes weekends, and how unmapped columns like `day_of_week_name` are automatically prefixed.

In [0]:
print("--- Querying 'rp_order_date' view ---")
# Note the new, auto-prefixed columns like 'order_date_day_of_week_name'
spark.table("rp_order_date").show()

print("--- Querying 'rp_ship_date' view (filtered for weekdays) ---")
spark.table("rp_ship_date").show()

--- Querying 'rp_order_date' view ---
+--------------+----------+---------------------------+---------------------+---------------------+
|order_date_key|order_date|order_date_day_of_week_name|order_date_is_weekend|order_date_is_holiday|
+--------------+----------+---------------------------+---------------------+---------------------+
|      20250101|2025-01-01|                  Wednesday|                false|                 true|
|      20250102|2025-01-02|                   Thursday|                false|                false|
|      20250103|2025-01-03|                     Friday|                false|                false|
|      20250104|2025-01-04|                   Saturday|                 true|                false|
|      20250105|2025-01-05|                     Sunday|                 true|                false|
|      20250106|2025-01-06|                     Monday|                false|                false|
|      20250107|2025-01-07|                    Tuesday|       

### 5. Summary

The utility successfully created distinct, aliased, and filtered views from a single base dimension. The schema-aware aliasing makes the solution robust to schema evolution, reducing maintenance overhead.